# Modelagem Supervisionada — Baseline Pré-Lançamento

**Grupo 6 | PISI 3 | 2026.1**

Este notebook implementa a classificação de sucesso comercial de jogos na Steam usando **apenas atributos disponíveis antes ou no momento do lançamento**.

**Target:** `commercial_success = 1` para jogos no top 20% de `user_reviews` (proxy de alcance observado). `user_reviews` é usada **apenas** para criar o target, **nunca como feature**.

**Features permitidas (pré-lançamento):**
- `price_final`, `discount`, `required_age`
- `release_year`, `release_month`
- `steam_deck`, `win`, `mac`, `linux`

**Variáveis proibidas no X (pós-lançamento):**
`avg_hours`, `median_hours`, `positive_ratio`, `rating`, `user_reviews`, `reviews`, `total_reviews`, `est_revenue_proxy`, `revenue_proxy`, `commercial_success`.

> O notebook contém uma **célula de auditoria obrigatória** que interrompe a execução caso alguma variável proibida seja detectada no conjunto de features.

## 1. Imports e Configuração

In [ ]:
!pip install imbalanced-learn shap -q

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve
)
from sklearn.utils import resample

import shap
import joblib

warnings.filterwarnings('ignore')
shap.initjs()
print(f"✓ SHAP versão: {shap.__version__}")
print("✓ Todas as bibliotecas importadas com sucesso.")

## 2. Carregamento dos Dados Processados

In [ ]:
# Carregamento dos Dados para Modelagem
# Os dados são gerados pelo notebook steam_preprocessing.ipynb
# Execute aquele notebook primeiro caso o arquivo não exista.

data = np.load('datasets_modelagem_final.npz', allow_pickle=True)

X_train = data['X_train'].astype(float)
X_test  = data['X_test'].astype(float)
y_train = data['y_train']
y_test  = data['y_test']
feature_names = data['feature_names']

print("Dados carregados com sucesso.")
print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"Features: {list(feature_names)}")
print(f"\nDistribuição y_train — 0: {(y_train==0).sum()} | 1: {(y_train==1).sum()}")
print(f"Distribuição y_test  — 0: {(y_test==0).sum()}  | 1: {(y_test==1).sum()}")

## 3. Auditoria de Features (Obrigatória)

Esta célula verifica se alguma variável pós-lançamento entrou acidentalmente no conjunto de features. A execução é interrompida com erro se isso ocorrer.

In [ ]:
# AUDITORIA OBRIGATÓRIA: verifica se há variáveis pós-lançamento no X
termos_proibidos = [
    'rating', 'review', 'reviews', 'user_reviews', 'avg_hours', 'hours',
    'positive_ratio', 'revenue', 'est_revenue', 'commercial_success'
]
cols_proibidas_encontradas = [
    col for col in feature_names
    if any(termo in col.lower() for termo in termos_proibidos)
]

print("=== AUDITORIA DE FEATURES ===")
print(f"Features no X: {list(feature_names)}")

if cols_proibidas_encontradas:
    raise ValueError(
        f"\n❌ PIPELINE INVÁLIDO — Variáveis pós-lançamento detectadas no X:\n"
        f"   {cols_proibidas_encontradas}\n"
        f"   Remova essas colunas antes de continuar."
    )
else:
    print("✓ Conjunto de features adequado para o cenário de pré-lançamento.")
    print("  Nenhuma variável pós-lançamento encontrada no X.")

## 4. Treinamento dos Modelos — Regressão Logística, Random Forest e Gradient Boosting

In [ ]:
# Treinamento dos Modelos

modelos = {
    "Regressão Logística": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

modelos_treinados = {}
for nome, modelo in modelos.items():
    print(f"Treinando: {nome}...")
    modelo.fit(X_train, y_train)
    modelos_treinados[nome] = modelo
    print(f"  ✓ Concluído")

## 5. Avaliação dos Modelos Iniciais

In [ ]:
# Avaliação — Métricas Comparativas

resultados = []

for nome, modelo in modelos_treinados.items():
    y_pred  = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    resultados.append({
        "Modelo":     nome,
        "Acurácia":   round(accuracy_score(y_test, y_pred), 4),
        "Precisão":   round(precision_score(y_test, y_pred), 4),
        "Recall":     round(recall_score(y_test, y_pred), 4),
        "F1-Score":   round(f1_score(y_test, y_pred), 4),
        "ROC-AUC":    round(roc_auc_score(y_test, y_proba), 4),
    })

df_resultados = pd.DataFrame(resultados).set_index("Modelo")
print("=== Tabela Comparativa de Desempenho ===")
display(df_resultados.sort_values("ROC-AUC", ascending=False))

In [ ]:
# Curvas ROC — Comparação entre Modelos

fig_roc = go.Figure()

for nome, modelo in modelos_treinados.items():
    y_proba = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)

    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f"{nome} (AUC = {auc:.3f})"
    ))

fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    line=dict(dash='dash', color='gray'),
    name='Baseline (Aleatório)'
))

fig_roc.update_layout(
    title="Curvas ROC — Comparação entre Modelos",
    xaxis_title="Taxa de Falsos Positivos (FPR)",
    yaxis_title="Taxa de Verdadeiros Positivos (TPR)",
    template="plotly_white",
    font=dict(size=12, family="Times New Roman"),
    legend=dict(x=0.6, y=0.1)
)
fig_roc.show()

In [ ]:
# Matrizes de Confusão

for nome, modelo in modelos_treinados.items():
    y_pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    fig_cm = px.imshow(
        cm,
        text_auto=True,
        color_continuous_scale="Blues",
        labels=dict(x="Predição", y="Real", color="Contagem"),
        x=["Insucesso (0)", "Sucesso (1)"],
        y=["Insucesso (0)", "Sucesso (1)"],
        title=f"Matriz de Confusão — {nome}",
        template="plotly_white"
    )
    fig_cm.update_layout(font=dict(size=12, family="Times New Roman"))
    fig_cm.show()

In [ ]:
# Importância de Features — Melhor Modelo

melhor_nome = df_resultados["ROC-AUC"].idxmax()
melhor_modelo = modelos_treinados[melhor_nome]

print(f"Melhor modelo: {melhor_nome}")

if hasattr(melhor_modelo, "feature_importances_"):
    importancias = melhor_modelo.feature_importances_
else:
    importancias = np.abs(melhor_modelo.coef_[0])

df_imp = pd.DataFrame({
    "Feature":    feature_names,
    "Importância": importancias
}).sort_values("Importância", ascending=True)

fig_imp = px.bar(
    df_imp,
    x="Importância",
    y="Feature",
    orientation="h",
    title=f"Importância das Features — {melhor_nome}",
    labels={"Importância": "Importância Relativa", "Feature": "Atributo"},
    template="plotly_white",
    color="Importância",
    color_continuous_scale="Blues"
)
fig_imp.update_layout(
    showlegend=False,
    font=dict(size=12, family="Times New Roman"),
    coloraxis_showscale=False
)
fig_imp.show()

## 6. Modelo SVM (kernel linear)

In [ ]:
# Treinamento — SVM (kernel linear)

print("Treinando: SVM (kernel linear)...")
print("  ⏳ Pode levar alguns minutos no Colab dependendo do ambiente...")

svm_model = SVC(
    kernel='linear',
    probability=True,   # Necessário para ROC-AUC via predict_proba
    random_state=42,
    C=1.0               # Regularização padrão (sem tuning)
)

svm_model.fit(X_train, y_train)
modelos_treinados["SVM (Linear)"] = svm_model

print("  ✓ SVM treinado com sucesso.")

In [ ]:
# Avaliação Completa — SVM

y_pred_svm  = svm_model.predict(X_test)
y_proba_svm = svm_model.predict_proba(X_test)[:, 1]

acc_svm  = accuracy_score(y_test, y_pred_svm)
prec_svm = precision_score(y_test, y_pred_svm)
rec_svm  = recall_score(y_test, y_pred_svm)
f1_svm   = f1_score(y_test, y_pred_svm)
auc_svm  = roc_auc_score(y_test, y_proba_svm)

print("=" * 55)
print("         MÉTRICAS GERAIS — SVM (Linear)")
print("=" * 55)
print(f"  Acurácia  : {acc_svm:.4f}")
print(f"  ROC-AUC   : {auc_svm:.4f}")
print("-" * 55)
print("  ★ Métricas da Classe 1 (Sucesso Comercial):")
print(f"    Precisão  (classe 1): {prec_svm:.4f}")
print(f"    Recall    (classe 1): {rec_svm:.4f}")
print(f"    F1-Score  (classe 1): {f1_svm:.4f}")
print("=" * 55)
print("\nRelatório Completo:\n")
print(classification_report(y_test, y_pred_svm,
                             target_names=["Insucesso (0)", "Sucesso (1)"]))

In [ ]:
# Tabela Comparativa Atualizada (com SVM)

resultados_atualizados = []

for nome, modelo in modelos_treinados.items():
    yp    = modelo.predict(X_test)
    yprob = modelo.predict_proba(X_test)[:, 1]

    resultados_atualizados.append({
        "Modelo":             nome,
        "Acurácia":           round(accuracy_score(y_test, yp), 4),
        "Precisão (cls 1)":   round(precision_score(y_test, yp), 4),
        "Recall (cls 1)":     round(recall_score(y_test, yp), 4),
        "F1-Score (cls 1)":   round(f1_score(y_test, yp), 4),
        "ROC-AUC":            round(roc_auc_score(y_test, yprob), 4),
    })

df_resultados_final = (
    pd.DataFrame(resultados_atualizados)
    .set_index("Modelo")
    .sort_values("ROC-AUC", ascending=False)
)

print("=== Tabela Comparativa Final — Todos os Modelos ===")
display(df_resultados_final)

In [ ]:
# Curvas ROC — 4 Modelos

fig_roc_final = go.Figure()

for nome, modelo in modelos_treinados.items():
    yprob      = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, yprob)
    auc         = roc_auc_score(y_test, yprob)

    fig_roc_final.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f"{nome} (AUC = {auc:.3f})"
    ))

fig_roc_final.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    line=dict(dash='dash', color='gray'),
    name='Baseline (Aleatório)'
))

fig_roc_final.update_layout(
    title="Curvas ROC — Comparação Final (4 Modelos)",
    xaxis_title="Taxa de Falsos Positivos (FPR)",
    yaxis_title="Taxa de Verdadeiros Positivos (TPR)",
    template="plotly_white",
    font=dict(size=12, family="Times New Roman"),
    legend=dict(x=0.55, y=0.1)
)
fig_roc_final.show()

In [ ]:
# Matriz de Confusão — SVM

cm_svm = confusion_matrix(y_test, y_pred_svm)

fig_cm_svm = px.imshow(
    cm_svm,
    text_auto=True,
    color_continuous_scale="Blues",
    labels=dict(x="Predição", y="Real", color="Contagem"),
    x=["Insucesso (0)", "Sucesso (1)"],
    y=["Insucesso (0)", "Sucesso (1)"],
    title="Matriz de Confusão — SVM (Linear)",
    template="plotly_white"
)
fig_cm_svm.update_layout(font=dict(size=12, family="Times New Roman"))
fig_cm_svm.show()

In [ ]:
# Salvamento — SVM e Tabela de Métricas

joblib.dump(svm_model, 'modelo_svm_linear.pkl')
print("✓ Modelo SVM salvo em: modelo_svm_linear.pkl")

df_resultados_final.to_csv('metricas_comparativas_modelos.csv')
print("✓ Tabela de métricas salva em: metricas_comparativas_modelos.csv")

svm_recarregado = joblib.load('modelo_svm_linear.pkl')
print(f"✓ Verificação: modelo recarregado — {type(svm_recarregado).__name__}")

In [ ]:
# Análise Final — Comparação e Recomendações

melhor_geral    = df_resultados_final["ROC-AUC"].idxmax()
melhor_auc      = df_resultados_final["ROC-AUC"].max()
auc_gb          = df_resultados_final.loc["Gradient Boosting", "ROC-AUC"] \
                  if "Gradient Boosting" in df_resultados_final.index else None
auc_svm_val     = df_resultados_final.loc["SVM (Linear)", "ROC-AUC"]

print("=" * 60)
print("         SÍNTESE FINAL DA COMPARAÇÃO DE MODELOS")
print("=" * 60)

print(f"\n1. MÉTRICAS DO SVM (Linear):")
for col in df_resultados_final.columns:
    val = df_resultados_final.loc["SVM (Linear)", col]
    print(f"   {col:<22}: {val:.4f}")

print(f"\n2. O SVM SUPEROU ALGUM MODELO?")
for nome in df_resultados_final.index:
    if nome == "SVM (Linear)":
        continue
    auc_outro = df_resultados_final.loc[nome, "ROC-AUC"]
    resultado = "✓ SVM supera" if auc_svm_val > auc_outro else "✗ SVM não supera"
    print(f"   {resultado} {nome} (AUC: {auc_svm_val:.4f} vs {auc_outro:.4f})")

print(f"\n3. GRADIENT BOOSTING CONTINUA SENDO O MELHOR?")
if melhor_geral == "Gradient Boosting":
    print(f"   ✓ Sim — Gradient Boosting lidera com ROC-AUC = {melhor_auc:.4f}")
else:
    print(f"   ✗ Não — o melhor modelo agora é: {melhor_geral} (AUC = {melhor_auc:.4f})")

print(f"\n4. MODELO RECOMENDADO PARA SHAP: Gradient Boosting")
print("=" * 60)

## 7. Interpretabilidade com SHAP

Uso do `TreeExplainer` sobre o Gradient Boosting para explicar as predições e identificar os fatores mais importantes na previsão de sucesso comercial pré-lançamento.

In [ ]:
print(f"✓ SHAP versão: {shap.__version__} — pronto para uso.")

print("Modelos disponíveis em 'modelos_treinados':")
for nome in modelos_treinados.keys():
    print(f"  • {nome}")

gb_model = modelos_treinados["Gradient Boosting"]
print(f"\n✓ Modelo selecionado: {type(gb_model).__name__}")

In [ ]:
# Preparação dos Dados para SHAP

X_test_df = pd.DataFrame(X_test, columns=feature_names)

X_shap, y_shap = resample(
    X_test_df, y_test,
    n_samples=2000,
    random_state=42,
    stratify=y_test
)

print(f"✓ X_test_df shape  : {X_test_df.shape}")
print(f"✓ Amostra SHAP     : {X_shap.shape}")
print(f"  Classe 0: {(y_shap==0).sum()} | Classe 1: {(y_shap==1).sum()}")

In [ ]:
# Cálculo dos SHAP Values — TreeExplainer

explainer = shap.TreeExplainer(gb_model)

print("Calculando SHAP values... (pode levar ~30s)")
shap_values = explainer.shap_values(X_shap)

print(f"✓ shap_values shape: {np.array(shap_values).shape}")

if isinstance(explainer.expected_value, np.ndarray):
    expected_val = explainer.expected_value[0]
else:
    expected_val = explainer.expected_value

print(f"✓ expected_value   : {expected_val:.4f}")

In [ ]:
# SHAP Summary Plot — Impacto e Direção

plt.figure()
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    plot_type="dot",
    show=False,
    plot_size=(10, 7)
)
plt.title("SHAP Summary Plot — Gradient Boosting\nImpacto das Features na Predição de Sucesso Comercial",
          fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("shap_summary_plot.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Salvo: shap_summary_plot.png")

In [ ]:
# SHAP Bar Plot — Importância Média Absoluta

plt.figure()
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    plot_size=(10, 7)
)
plt.title("SHAP Bar Plot — Importância Média Absoluta por Feature\nGradient Boosting | Sucesso Comercial de Jogos",
          fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("shap_bar_plot.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Salvo: shap_bar_plot.png")

In [ ]:
# SHAP Dependence Plots — Top 3 Features

mean_abs_shap = np.abs(shap_values).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[::-1][:3]
top3_features = [feature_names[i] for i in top3_idx]

print(f"Top 3 features para Dependence Plots: {top3_features}\n")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feat in enumerate(top3_features):
    plt.sca(axes[i])
    shap.dependence_plot(
        feat,
        shap_values,
        X_shap,
        ax=axes[i],
        show=False
    )
    axes[i].set_title(f"Dependence Plot\n{feat}", fontsize=11)

plt.suptitle("SHAP Dependence Plots — Top 3 Features | Gradient Boosting",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("shap_dependence_plots.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Salvo: shap_dependence_plots.png")

In [ ]:
# Ranking de Importância SHAP

mean_abs_shap = np.abs(shap_values).mean(axis=0)

df_shap_ranking = pd.DataFrame({
    "Feature":              feature_names,
    "SHAP Importância Média": mean_abs_shap
}).sort_values("SHAP Importância Média", ascending=False).reset_index(drop=True)

df_shap_ranking.index += 1
df_shap_ranking.index.name = "Ranking"

df_shap_ranking["Contribuição Relativa (%)"] = (
    df_shap_ranking["SHAP Importância Média"] /
    df_shap_ranking["SHAP Importância Média"].sum() * 100
).round(2)

print("=== Ranking de Importância SHAP — Gradient Boosting ===\n")
display(df_shap_ranking)

df_shap_ranking.to_csv("shap_ranking_features.csv")
print("\n✓ Salvo: shap_ranking_features.csv")

In [ ]:
# Salvamento — Gradient Boosting

joblib.dump(gb_model, 'modelo_gradient_boosting.pkl')
print("✓ Modelo Gradient Boosting salvo em: modelo_gradient_boosting.pkl")

gb_recarregado = joblib.load('modelo_gradient_boosting.pkl')
y_check = gb_recarregado.predict(X_test[:5])
print(f"✓ Verificação OK — primeiras 5 predições: {y_check}")

In [ ]:
# ==============================================================================
# SÍNTESE DE INTERPRETABILIDADE SHAP — BASELINE PRÉ-LANÇAMENTO
# ==============================================================================
import pandas as pd
import numpy as np

# 1. Ranking SHAP com as features reais do modelo pré-lançamento
try:
    mean_shap_features = np.abs(shap_values).mean(axis=0)
    df_shap_ranking = pd.DataFrame({
        "Feature": feature_names,
        "SHAP Importância Média": mean_shap_features
    }).sort_values(by="SHAP Importância Média", ascending=False).reset_index(drop=True)

    total_shap = df_shap_ranking["SHAP Importância Média"].sum()
    df_shap_ranking["Contribuição Relativa (%)"] = (
        df_shap_ranking["SHAP Importância Média"] / total_shap * 100
    ).round(2)
except Exception as e:
    print(f"Aviso ao gerar dataframe do SHAP: {e}")

# 2. Top 3
top1 = df_shap_ranking.iloc[0] if len(df_shap_ranking) > 0 else {"Feature": "N/A", "SHAP Importância Média": 0, "Contribuição Relativa (%)": 0}
top2 = df_shap_ranking.iloc[1] if len(df_shap_ranking) > 1 else {"Feature": "N/A", "SHAP Importância Média": 0, "Contribuição Relativa (%)": 0}
top3 = df_shap_ranking.iloc[2] if len(df_shap_ranking) > 2 else {"Feature": "N/A", "SHAP Importância Média": 0, "Contribuição Relativa (%)": 0}

def rank_of(feat_name):
    matches = df_shap_ranking[df_shap_ranking["Feature"] == feat_name].index
    return matches[0] + 1 if len(matches) > 0 else "Não encontrado"

rank_price    = rank_of("price_final")
rank_discount = rank_of("discount")
rank_sd       = rank_of("steam_deck")
rank_year     = rank_of("release_year")

# 4. Síntese
print("=" * 65)
print("   SÍNTESE DE INTERPRETABILIDADE SHAP — BASELINE PRÉ-LANÇAMENTO")
print("   Pergunta de Pesquisa: Fatores de Sucesso Comercial (PP2)")
print("   Modelo: Gradient Boosting | Features: apenas pré-lançamento")
print("=" * 65)

print(f"""
A) VARIÁVEIS MAIS IMPORTANTES SEGUNDO O SHAP (pré-lançamento):
   1º → {top1['Feature']}  (SHAP médio: {top1['SHAP Importância Média']:.4f} | {top1['Contribuição Relativa (%)']:.1f}%)
   2º → {top2['Feature']}  (SHAP médio: {top2['SHAP Importância Média']:.4f} | {top2['Contribuição Relativa (%)']:.1f}%)
   3º → {top3['Feature']}  (SHAP médio: {top3['SHAP Importância Média']:.4f} | {top3['Contribuição Relativa (%)']:.1f}%)

B) VARIÁVEIS EXCLUÍDAS INTENCIONALMENTE DO MODELO (pós-lançamento):
   → avg_hours, positive_ratio, rating, user_reviews e est_revenue_proxy
     foram removidas porque só existem APÓS o lançamento do jogo.

C) price_final ocupa o ranking #{rank_price}.
D) discount ocupa o ranking #{rank_discount}.
E) steam_deck ocupa o ranking #{rank_sd}.
F) release_year ocupa o ranking #{rank_year}.
""")
print("=" * 65)
print("✓ Síntese pré-lançamento concluída.")
print("=" * 65)